# SNI-21 — frozen A0 per source domain

Evaluasi checkpoint A0 yang sama pada validation Adrian dan Faruq. Notebook ini tidak training dan tidak memulihkan test.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil, subprocess, sys
from pathlib import Path

REPO = Path('/content/coffee-bean-detection')
BRANCH = 'agent/add-vadcp-pipeline'
os.chdir('/content')
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run([
    'git', 'clone', '--depth', '1', '--branch', BRANCH,
    'https://github.com/ediprin/coffee-bean-detection.git', str(REPO),
], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
for module_name in list(sys.modules):
    if module_name == 'coffee_detector' or module_name.startswith('coffee_detector.'):
        sys.modules.pop(module_name, None)
sys.path.insert(0, str(REPO / 'src'))
os.chdir(REPO)
import coffee_detector
print('IMPORT:', coffee_detector.__file__)


In [ ]:
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root

PROJECT_ROOT = resolve_drive_project_root()
A0_ARCHIVE = require_project_artifact(
    PROJECT_ROOT, 'bundles/sni21-vadcp-pilot-bundle/A0_real.tar'
)
CHECKPOINT = require_project_artifact(
    PROJECT_ROOT, 'checkpoints/sni21-vadcp-pilot-results/A0_seed42/weights/best.pt'
)

COMBINED_ROOT = Path('/content/sni21-a0-development')
SEPARATED_ROOT = Path('/content/sni21-source-separated-v1')
OUTPUT_ROOT = PROJECT_ROOT / 'experiments/sni21-source-domain-evaluation-v1'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('ARCHIVE   :', A0_ARCHIVE)
print('CHECKPOINT:', CHECKPOINT)
print('OUTPUT    :', OUTPUT_ROOT)


In [ ]:
from coffee_detector.archive_sni21_pilot import restore_real_a0_development
from coffee_detector.separate_sni21_sources import separate_sni21_sources

restore_real_a0_development(A0_ARCHIVE, COMBINED_ROOT)
assert not (COMBINED_ROOT / 'test').exists()
separation = separate_sni21_sources(COMBINED_ROOT, SEPARATED_ROOT, link_mode='auto')
assert separation['training_executed'] is False
assert separation['test_images_accessed'] is False
print('PEMISAHAN SIAP')


In [ ]:
import json
import torch
from coffee_detector.evaluate_sni21_source_domains import evaluate_sni21_source_domains

DEVICE = '0' if torch.cuda.is_available() else 'cpu'
print('DEVICE:', DEVICE, '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
summary = evaluate_sni21_source_domains(
    CHECKPOINT, SEPARATED_ROOT, OUTPUT_ROOT, device=DEVICE
)
assert summary['training_executed'] is False
assert summary['test_images_accessed'] is False
print(json.dumps(summary, indent=2, ensure_ascii=False))


In [ ]:
import pandas as pd
from IPython.display import display

table = pd.DataFrame(summary['rows'])
percent = ['map50_95', 'map50', 'precision', 'recall', 'macro_map50_95', 'bottom3_map50_95', 'worst_map50_95']
display(table.style.format({name: '{:.2%}' for name in percent}))
print('TRAINING:', summary['training_executed'])
print('TEST ACCESSED:', summary['test_images_accessed'])
print('SUMMARY:', summary['summary'])
print('Kirim tabel ini. Jangan training model baru.')


In [ ]:
import importlib
import coffee_detector.analyze_sni21_source_classes as class_audit_module
importlib.reload(class_audit_module)

class_audit = class_audit_module.analyze_sni21_source_classes(
    SEPARATED_ROOT, summary['summary'], OUTPUT_ROOT / 'class_audit'
)
for source, result in class_audit['sources'].items():
    print(f'\n=== {source}: BOTTOM-5 ===')
    display(pd.DataFrame(result['bottom5']).style.format({'map50_95': '{:.2%}'}))
    print('Pergeseran prevalensi train->val terbesar:')
    display(pd.DataFrame(result['largest_train_val_prevalence_shifts'])[[
        'class_name', 'train_instances', 'val_instances', 'val_to_train_prevalence_ratio'
    ]])
    print('Missing train GT:', result['classes_without_train_ground_truth'])
    print('Missing GT:', result['classes_without_ground_truth'])
    print('Korelasi log-support/AP:', result['log_support_ap_correlation'])
print('\nSHARED BOTTOM-5:', class_audit['shared_bottom5_classes'])
print('TRAINING:', class_audit['training_executed'])
print('TEST ACCESSED:', class_audit['test_images_accessed'])
print('SUMMARY:', class_audit['summary'])
print('Kirim dua tabel Bottom-5 dan shared classes.')


In [ ]:
import coffee_detector.run_sni21_hard_class_visual_audit as visual_audit_module
importlib.reload(visual_audit_module)
from IPython.display import Image as DisplayImage

visual_audit = visual_audit_module.run_sni21_hard_class_visual_audit(
    SEPARATED_ROOT,
    class_audit['summary'],
    OUTPUT_ROOT / 'hard_class_visual_audit',
    samples_per_split=8,
    seed=42,
)
assert visual_audit['training_executed'] is False
assert visual_audit['inference_executed'] is False
assert visual_audit['test_images_accessed'] is False
for row in visual_audit['sheets']:
    print(f"\n{row['source_dataset']} / {row['class_name']}")
    display(DisplayImage(filename=row['contact_sheet']))
print('SUMMARY:', visual_audit['summary'])
print('Kirim contact sheet kelas yang label/visualnya tampak tidak konsisten.')
